<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/TCS_Differential_Process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import pandas as pd
tcs_backend_folder = "/content/drive/MyDrive/Data/UCOP_Job_Codes_Summary"
shift_on_call_full_df = pd.read_excel(f"{tcs_backend_folder}/UPTE-26-015_Job Summary Report_3.20.26.xlsx", sheet_name="Shift and On Call Rates")
shift_on_call_full_df = shift_on_call_full_df.iloc[:-2]
shift_on_call_full_df.shape

(37047, 11)

In [42]:
shift_on_call_full_df.tail()

,Salary Plan SETID,Job Code,Job Description,Pay Representation,Effective Date,Effective Status,Earn Code,Earn Code Description,Shift On Call Type,Hourly Rate,Earnings Code
37042,UCOP1,008211,BLDG MAINT WORKER LD,C,2024-06-23,A,NSD,Night Shift Differential *,SD03,1.0,
37043,UCOP1,008212,BLDG MAINT WORKER SR,C,2024-06-23,A,ESD,Evening Shift Differential,SD02,1.0,
37044,UCOP1,008212,BLDG MAINT WORKER SR,C,2024-06-23,A,NSD,Night Shift Differential *,SD03,1.0,
37045,UCOP1,008213,BLDG MAINT WORKER,C,2024-06-23,A,ESD,Evening Shift Differential,SD02,1.0,
37046,UCOP1,008213,BLDG MAINT WORKER,C,2024-06-23,A,NSD,Night Shift Differential *,SD03,1.0,


In [44]:
job_code_review_full_df = pd.read_excel(f"{tcs_backend_folder}/UPTE-26-015_Job Summary Report_3.20.26.xlsx", sheet_name="Job Code Review")
job_code_review_full_df.shape

(46081, 48)

In [45]:
upte_units = ['HX', 'RX', 'TX']
job_code_review_upte_df = job_code_review_full_df[job_code_review_full_df['Union Code'].isin(upte_units)]
job_code_review_upte_df.shape

(4513, 48)

In [6]:
job_code_review_upte_df.columns

Index(['Salary Plan SETID', 'Jobcode SETID', 'Job Code', 'Eff Date',
       'Salary Grade Eff Status', 'Descr', 'Short Descr', 'Job Function',
       'Sal Admin Plan', 'Grade', 'Step', 'Manager Level', 'Survey Salary',
       'Union Code', 'Retro Rate', 'Retro Percent', 'Currency', 'Stnd Hrs/Wk',
       'Work Period', 'Comp Frequency', 'Job Family', 'Reg/Temp', 'Tipped',
       'Med Checkup', 'FLSA Status', 'EEO-1 Cat', 'EEO-4 Cat', 'EEO-5 Cat',
       'EEO-6 Cat', 'SOC', 'IPEDSS Code', 'OCC', 'Telework', 'Company',
       'Encmb Indc', 'Posn Mgmt?', 'Enc Sal Optn', 'Enc Sal Amt',
       'Last Update ', 'Last Updated by', 'Last Updated Date Time',
       'Key Job Code', 'CTO/OS Subgroup', 'Sys/Local Ind', 'OnCall Eligible',
       'Elig Shift Diff', 'Classified', 'OSHPD Code'],
      dtype='object')

In [20]:
shift_on_call_full_df.columns

Index(['Salary Plan SETID', 'Job Code', 'Job Description',
       'Pay Representation', 'Effective Date', 'Effective Status', 'Earn Code',
       'Earn Code Description', 'Shift On Call Type', 'Hourly Rate',
       'Earnings Code'],
      dtype='object')

In [46]:
upte_shift_on_call_df = shift_on_call_full_df.merge(job_code_review_upte_df[['Job Code', 'Union Code']], on= 'Job Code', how='inner').drop_duplicates()
upte_shift_on_call_df.shape

(6757, 12)

In [40]:
output_path = "/content/drive/MyDrive/UPTE_Differential_lists"
upte_shift_on_call_df.to_excel(f"{output_path}/upte_systemwide_diff_master_list_0320206.xlsx", index=False)

In [47]:
business_units = upte_shift_on_call_df['Salary Plan SETID'].unique()
business_units

array(['BKCMP', 'DVCMP', 'DVMED', 'IRCMP', 'IRMED', 'LACMP', 'LAMED',
       'LBNL1', 'MECMP', 'RVCMP', 'SBCMP', 'SCCMP', 'SDCMP', 'SDMED',
       'SFCMP', 'SFMED', 'UCANR'], dtype=object)

In [48]:
diff_types_list = upte_shift_on_call_df['Earn Code Description'].unique()
diff_types_list

array(['Evening Shift Differential', 'Night Shift Differential *',
       'Time On Call', 'Non Prod Evening Shift Diff',
       'Non Prod Night Shift Diff', 'Non Prod Weekend  Shift Diff',
       'Weekend Day Shift Differential', 'Weekend Eve Shift Differential',
       'Weekend Night Shift Differentl', 'NRA Charge Differential'],
      dtype=object)

Process all differential details at the bargaining unit and job title level into dictionaries

In [58]:

diff_details_dict = {}

for bus_unit in business_units:

  curr_bus_unit_df = upte_shift_on_call_df[upte_shift_on_call_df['Salary Plan SETID'] == bus_unit]

  diff_details_dict[bus_unit] = {}

  for barg_unit in curr_bus_unit_df['Union Code'].unique():
    diff_details_dict[bus_unit][barg_unit] = {}

    curr_barg_unit_df = curr_bus_unit_df[curr_bus_unit_df['Union Code'] == barg_unit]
    titles_list = sorted(curr_barg_unit_df['Job Description'].unique())

    for title in titles_list:

      diff_details_dict[bus_unit][barg_unit][title] = {}
      diff_details_dict[bus_unit][barg_unit][title]["Job Title"] = title
      diff_details_dict[bus_unit][barg_unit][title]["Job Code"] = \
      curr_barg_unit_df[curr_barg_unit_df['Job Description'] == title]['Job Code'].values[0]

    # process the differential types
      for diff_type in diff_types_list:
        filter = (curr_barg_unit_df['Earn Code Description'] == diff_type) & \
                (curr_barg_unit_df['Job Description'] == title)
        if not curr_barg_unit_df[filter].empty:
          r = curr_barg_unit_df[filter]['Hourly Rate'].values[0]
          diff_details_dict[bus_unit][barg_unit][title][diff_type] = f'${r:,.2f}'
        else:
          diff_details_dict[bus_unit][barg_unit][title][diff_type] = 0



In [59]:
diff_details_dict

{'BKCMP': {'TX': {'ANML HEALTH TCHN 1': {'Job Title': 'ANML HEALTH TCHN 1',
    'Job Code': '009537',
    'Evening Shift Differential': '$0.72',
    'Night Shift Differential *': '$0.67',
    'Time On Call': 0,
    'Non Prod Evening Shift Diff': 0,
    'Non Prod Night Shift Diff': 0,
    'Non Prod Weekend  Shift Diff': 0,
    'Weekend Day Shift Differential': 0,
    'Weekend Eve Shift Differential': 0,
    'Weekend Night Shift Differentl': 0,
    'NRA Charge Differential': 0},
   'ANML HEALTH TCHN 2': {'Job Title': 'ANML HEALTH TCHN 2',
    'Job Code': '009536',
    'Evening Shift Differential': '$0.67',
    'Night Shift Differential *': '$0.67',
    'Time On Call': 0,
    'Non Prod Evening Shift Diff': 0,
    'Non Prod Night Shift Diff': 0,
    'Non Prod Weekend  Shift Diff': 0,
    'Weekend Day Shift Differential': 0,
    'Weekend Eve Shift Differential': 0,
    'Weekend Night Shift Differentl': 0,
    'NRA Charge Differential': 0},
   'ANML HEALTH TCHN 3': {'Job Title': 'ANML HEALTH

Create the output excel files

In [57]:
# create the folders (only once)
# DONE!!! NEVER USE IT AGAIN
import os
output_path = "/content/drive/MyDrive/UPTE_Differential_lists"
for bus_unit in business_units:
  os.makedirs(f"{output_path}/{bus_unit}", exist_ok=True)

In [61]:
for bus_unit in business_units:

  barg_units = ['HX', 'RX', 'TX']

  with pd.ExcelWriter(f"{output_path}/{bus_unit}/{bus_unit}_differential_master_list_0320206.xlsx") as writer:


  # loop throug hteh bargainign units
    for barg_unit in diff_details_dict[bus_unit]:
      print(f"Processing {bus_unit}'s {barg_unit}")

      campus_bu = diff_details_dict[bus_unit][barg_unit]

      curr_df_list = []
      for title in diff_details_dict[bus_unit][barg_unit]:
        df = pd.DataFrame(campus_bu[title], index=[0])
        curr_df_list.append(df)

      curr_df = pd.concat(curr_df_list)
      curr_df.to_excel(writer, sheet_name=barg_unit, index=False)

Processing BKCMP's TX
Processing BKCMP's RX
Processing DVCMP's TX
Processing DVCMP's HX
Processing DVCMP's RX
Processing DVMED's TX
Processing DVMED's HX
Processing DVMED's RX
Processing IRCMP's TX
Processing IRCMP's HX
Processing IRCMP's RX
Processing IRMED's TX
Processing IRMED's HX
Processing LACMP's TX
Processing LACMP's HX
Processing LACMP's RX
Processing LAMED's TX
Processing LAMED's HX
Processing LAMED's RX
Processing LBNL1's RX
Processing LBNL1's TX
Processing MECMP's TX
Processing MECMP's HX
Processing MECMP's RX
Processing RVCMP's TX
Processing RVCMP's HX
Processing RVCMP's RX
Processing SBCMP's TX
Processing SBCMP's HX
Processing SBCMP's RX
Processing SCCMP's TX
Processing SCCMP's HX
Processing SCCMP's RX
Processing SDCMP's TX
Processing SDCMP's HX
Processing SDCMP's RX
Processing SDMED's TX
Processing SDMED's HX
Processing SDMED's RX
Processing SFCMP's TX
Processing SFCMP's HX
Processing SFCMP's RX
Processing SFMED's TX
Processing SFMED's HX
Processing SFMED's RX
Processing

In [55]:
test_dict = diff_details_dict['SFMED']['HX']['STF PHARMACIST 2']

In [56]:
test_df = pd.DataFrame(test_dict, index=[0])
test_df

,Job Title,Job Code,Evening Shift Differential,Night Shift Differential *,Time On Call,Non Prod Evening Shift Diff,Non Prod Night Shift Diff,Non Prod Weekend Shift Diff,Weekend Day Shift Differential,Weekend Eve Shift Differential,Weekend Night Shift Differentl,NRA Charge Differential
0,STF PHARMACIST 2,009247,$4.00,$10.00,0,$4.00,$10.00,0,0,0,0,$1.25


In [13]:
upte_shift_on_call_df.head(100)

,Salary Plan SETID,Job Code,Job Description,Pay Representation,Effective Date,Effective Status,Earn Code,Earn Code Description,Shift On Call Type,Hourly Rate,Earnings Code,Union Code
0,BKCMP,004031,LIFEGUARD,C,1930-01-01,A,ESD,Evening Shift Differential,SD02,0.67,,TX
14,BKCMP,004031,LIFEGUARD,C,1930-01-01,A,NSD,Night Shift Differential *,SD03,0.67,,TX
28,BKCMP,004812,COMPUTER OPR SR,C,1930-01-01,A,ESD,Evening Shift Differential,SD02,0.85,,TX
43,BKCMP,004812,COMPUTER OPR SR,C,1930-01-01,A,NSD,Night Shift Differential *,SD03,0.85,,TX
58,BKCMP,004812,COMPUTER OPR SR,C,1930-01-01,A,TOC,Time On Call,OCR1,5.00,,TX
...,...,...,...,...,...,...,...,...,...,...,...,...
1364,BKCMP,007174,DEV TCHN 1,C,1930-01-01,A,ESD,Evening Shift Differential,SD02,0.67,,TX
1375,BKCMP,007174,DEV TCHN 1,C,1930-01-01,A,NSD,Night Shift Differential *,SD03,0.67,,TX
1386,BKCMP,007684,EDITOR,C,1930-01-01,A,ESD,Evening Shift Differential,SD02,0.57,,TX
1408,BKCMP,007684,EDITOR,C,1930-01-01,A,NSD,Night Shift Differential *,SD03,0.57,,TX
